# 🧠 DatasheetIQ — Knowledge Graph Backend (Google Colab)

Runs the **FastAPI backend + Qwen 3.5 4B model** in Google Colab.

**Run cells in order. Do NOT skip any cell.**

---

## ⚠️ PHASE 1 — Clean Environment
Run this FIRST after every runtime restart.

In [ ]:
# CELL 1: Clean conflicting pre-installed packages
!pip uninstall -y fastapi starlette httpx uvicorn pydantic python-multipart 2>/dev/null
print("✅ Cleaned pre-installed packages")

## 📦 PHASE 2 — Install Dependencies (Pinned Versions)

In [ ]:
# CELL 2: Install core backend (PINNED)
!pip install fastapi==0.124.1 uvicorn==0.34.0 starlette==0.49.1 pydantic==2.12.0 pydantic-settings httpx==0.28.1 python-multipart==0.0.18
print("\n✅ Core backend installed")

In [ ]:
# CELL 3: Install PDF parsing + Neo4j + ngrok
!pip install PyMuPDF==1.24.0 pdfplumber==0.11.0 Pillow==10.4.0 neo4j==5.23.0 pyngrok
print("\n✅ PDF parsing + Neo4j + ngrok installed")

In [ ]:
# CELL 4: Install AI/ML (uses Colab's pre-installed torch)
!pip install transformers accelerate
print("\n✅ Transformers + Accelerate installed")

## ✅ PHASE 3 — Validate Installation

In [ ]:
# CELL 5: Validate — ALL imports must pass with zero errors
import fastapi
import uvicorn
import pydantic
import starlette
import httpx
import fitz  # PyMuPDF
import pdfplumber
import neo4j
import torch
import transformers

print(f"fastapi     : {fastapi.__version__}")
print(f"uvicorn     : {uvicorn.__version__}")
print(f"pydantic    : {pydantic.__version__}")
print(f"starlette   : {starlette.__version__}")
print(f"httpx       : {httpx.__version__}")
print(f"PyMuPDF     : {fitz.version}")
print(f"neo4j       : {neo4j.__version__}")
print(f"torch       : {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"GPU         : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
print("\n✅ ALL IMPORTS VALID — No conflicts")

## 📂 PHASE 4 — Mount Google Drive + Set Backend Path

In [ ]:
# CELL 6: Mount Google Drive and set backend path
import os
import sys

from google.colab import drive
drive.mount('/content/drive')

# ⚠️ SET YOUR BACKEND PATH HERE
# This should point to the folder CONTAINING the 'app/' directory
BACKEND_PATH = '/content/drive/MyDrive/Knowledge_graph/backend'

# Verify the path exists
if os.path.exists(BACKEND_PATH):
    os.chdir(BACKEND_PATH)
    if BACKEND_PATH not in sys.path:
        sys.path.insert(0, BACKEND_PATH)
    print(f"✅ Working directory: {os.getcwd()}")
    print(f"✅ sys.path includes: {BACKEND_PATH}")
else:
    print(f"❌ Path not found: {BACKEND_PATH}")
    print("   Update BACKEND_PATH above to match your Google Drive structure")

In [ ]:
# CELL 7: Verify backend files exist
import os

required_files = [
    'app/__init__.py',
    'app/main.py',
    'app/config.py',
    'app/models.py',
    'app/routers/__init__.py',
    'app/routers/upload.py',
    'app/routers/query.py',
    'app/services/__init__.py',
    'app/services/pdf_parser.py',
    'app/services/table_extractor.py',
    'app/services/content_detector.py',
    'app/services/graph_builder.py',
    'app/services/query_engine.py',
    'app/services/ai_client.py',
    'app/utils/__init__.py',
]

missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    print("❌ MISSING FILES — upload these before continuing:")
    for f in missing:
        print(f"   {f}")
else:
    print("✅ All backend files found")
    # Quick check: verify app.main has FastAPI instance
    with open('app/main.py', 'r') as f:
        content = f.read()
    if 'app = FastAPI(' in content:
        print("✅ FastAPI app instance found in app/main.py")
    else:
        print("⚠️  WARNING: 'app = FastAPI()' not found in app/main.py")

## 🔗 PHASE 5 — Configure Neo4j Connection

In [ ]:
# CELL 8: Set Neo4j credentials
import os

os.environ['NEO4J_URI'] = 'neo4j+s://xxxxxxxx.databases.neo4j.io'  # ← Your Aura URI
os.environ['NEO4J_USER'] = 'neo4j'                                  # ← Your username
os.environ['NEO4J_PASSWORD'] = 'your-password-here'                  # ← Your password
os.environ['QWEN_API_URL'] = ''                                      # Leave empty

print(f"NEO4J_URI      = {os.environ['NEO4J_URI']}")
print(f"NEO4J_USER     = {os.environ['NEO4J_USER']}")
print(f"NEO4J_PASSWORD = {'*' * len(os.environ['NEO4J_PASSWORD'])}")
print("\n✅ Environment variables set")

## 🤖 PHASE 6 — Load Qwen 3.5 4B Model

In [ ]:
# CELL 9: Load Qwen 3.5 4B
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3.5-4B"

print(f"Loading {MODEL_NAME}...")
print(f"GPU: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"\n✅ {MODEL_NAME} loaded successfully")
print(f"   Device: {model.device}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")

In [ ]:
# CELL 10: Define generation function + test
import json

SYSTEM_PROMPT = """You are a precise technical assistant for semiconductor datasheets.
You receive structured data extracted from a knowledge graph and must format it into
a clear, accurate natural language answer.

RULES:
1. ONLY use the data provided — DO NOT add any information.
2. If data is missing, say "not found in the datasheet".
3. Always include units when available.
4. Always include conditions/test conditions when available.
5. For tables, present data in a clean, readable format.
6. Never guess or approximate values.
"""

def generate_answer(query: str, data: dict, system_prompt: str = "") -> str:
    if not system_prompt:
        system_prompt = SYSTEM_PROMPT

    user_message = (
        f"User question: {query}\n\n"
        f"Knowledge Graph Data:\n{json.dumps(data, indent=2)}\n\n"
        f"Format a clear, precise answer using ONLY the data above."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
        )

    generated = outputs[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# Quick test
test = generate_answer(
    "What is the supply voltage?",
    {"parameter": "VCC", "min": "2.7", "max": "5.5", "unit": "V"}
)
print(f"✅ Qwen 3.5 4B test: {test}")

## 🚀 PHASE 7 — Patch ai_client + Start Server

In [ ]:
# CELL 11: Patch ai_client to call Qwen directly (no HTTP)
ai_client_code = '''
import logging

logger = logging.getLogger(__name__)

_generate_fn = None

def set_generate_function(fn):
    global _generate_fn
    _generate_fn = fn

async def format_with_qwen(query: str, structured_data: dict) -> str:
    if _generate_fn is None:
        logger.warning("Qwen model not loaded — returning raw data")
        return ""
    try:
        answer = _generate_fn(query, structured_data)
        return answer
    except Exception as e:
        logger.error("Qwen generation error: %s", e)
        return ""
'''

with open('app/services/ai_client.py', 'w') as f:
    f.write(ai_client_code)

print("✅ ai_client.py patched — Qwen runs in-process")

In [ ]:
# CELL 12: Wire Qwen model to backend
from app.services.ai_client import set_generate_function
set_generate_function(generate_answer)
print("✅ Qwen 3.5 4B wired to backend")

In [ ]:
# CELL 13: Start ngrok tunnel
# ⚠️ Get your FREE token: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # ← REPLACE THIS!

from pyngrok import ngrok

ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)

print("=" * 60)
print(f"🌐 PUBLIC BACKEND URL: {public_url}")
print("=" * 60)
print(f"  Health:  {public_url}/health")
print(f"  Upload:  {public_url}/api/upload")
print(f"  Query:   {public_url}/api/query")

In [ ]:
# CELL 14: Start FastAPI server (BLOCKS — run last!)
# ──────────────────────────────────────────────────────
# The import path is "app.main:app" because:
#   - We cd'd into the backend/ directory in Cell 6
#   - app/ is a package, main.py has the FastAPI instance
#   - So the module path from cwd is: app.main
#   - The variable name is: app
# ──────────────────────────────────────────────────────
import uvicorn

print(f"Working directory: {os.getcwd()}")
print("🚀 Starting FastAPI on port 8000...")
print("   Access via ngrok URL above.")
print("   Press STOP (■) to shut down.\n")

uvicorn.run("app.main:app", host="0.0.0.0", port=8000, log_level="info")